Small

In [ ]:
# ============================================================
# HAZMAT VRP FINAL (ROBUST VERSION)
# ============================================================

import pandas as pd
import pulp as pl
import math

# ============================================================
# 1. OD MATRIX LADEN & TRADE-OFF FILTER
# ============================================================



w_risk = 0.5  # w1 (Risiko)
w_cost = 0.3  # w2 (Kosten/Distanz)
w_time = 0.2  # w3 (Zeit)



od_df = pd.read_csv(r"C:\Users\j.beckmann\OneDrive - Reply\Uni Jonas Beckmann\4.Semester\Operations Research\Projekt\Umwelftfreundliche_Routenplanung\models\Data\OperationsResearch\od_matrix_hazmat_small.csv")
od_time_df = pd.read_csv(r"C:\Users\j.beckmann\OneDrive - Reply\Uni Jonas Beckmann\4.Semester\Operations Research\Projekt\Umwelftfreundliche_Routenplanung\models\Data\OperationsResearch\od_matrix_hazmat_small_time.csv")
cc_df = pd.read_csv(r"C:\Users\j.beckmann\OneDrive - Reply\Uni Jonas Beckmann\4.Semester\Operations Research\Projekt\Umwelftfreundliche_Routenplanung\models\Data\OperationsResearch\od_matrix_small_charging.csv")

# 1. RICHTIG: Distanz und Zeit für normale Kanten MERGEN
# Wir gehen davon aus, dass od_time_df die Spalten 'from', 'to' und 'time_min' hat.
od_merged_temp = od_df.merge(od_time_df[["from", "to", "time_min"]], on=["from", "to"], how="left")

# 2. Spalten für Charging Stations anpassen
cc_df = cc_df.rename(columns={"customer": "from", "charger": "to"})
cc_df_rev = cc_df.copy()
cc_df_rev = cc_df_rev.rename(columns={"from": "to", "to": "from"})


# 3. Ladestationen NICHT in den Trade-Off-Filter aufnehmen
od_all = od_merged_temp.copy()   # nur Kunden + Depot

# Ladestationen separat speichern
cc_all = pd.concat([cc_df, cc_df_rev], ignore_index=True)

# 4. Max-Werte für die Normalisierung finden
max_d = od_merged_temp["dist_km"].max()
max_r = od_merged_temp["risk_cost"].max()
max_t = od_merged_temp["time_min"].max()


max_d = max_d if pd.notna(max_d) and max_d > 0 else 1
max_r = max_r if pd.notna(max_r) and max_r > 0 else 1
max_t = max_t if pd.notna(max_t) and max_t > 0 else 1

# 5. Trade-Off Funktion
def get_eval_cost(row):
    d = row.get("dist_km", 99999)
    t = row.get("time_min", 99999)
    
    if pd.isna(d): d = 99999
    if pd.isna(t): t = 99999
    
    r = 0.0 if row["to"] == "DEPOT" else row.get("risk_cost", 0.0)
    if pd.isna(r): r = 0.0
    
    norm_dist = d / max_d
    norm_risk = r / max_r
    norm_time = t / max_t
    
    return (w_risk * norm_risk) + (w_cost * norm_dist) + (w_time * norm_time)

# Trade-Off nur für Kunden/Depot
od_all["eval_cost"] = od_all.apply(get_eval_cost, axis=1)
od_all = od_all.dropna(subset=['eval_cost'])

od_merged = od_all.loc[od_all.groupby(['from', 'to'])['eval_cost'].idxmin()]

# Ladestationen unverändert anhängen
od_merged = pd.concat([od_merged, cc_all], ignore_index=True)

print("Anzahl finaler Kanten nach Filter:", len(od_merged))

# ============================================================
# 2. DATENSTRUKTUREN (ROBUST!)
# ============================================================


dist, risk, time, cost = {}, {}, {}, {} 
nodes = set()

for _, row in od_merged.iterrows():
    i, j = row["from"], row["to"]

    d = row.get("dist_km", None)
    t = row.get("time_min", None)
    c = row.get("eval_cost", None)
    
    # Rückfahrt-Logik final in die Parameter übernehmen
    if j == "DEPOT":
        r = 0.0
    else:
        r = row.get("risk_cost", 0)
        if pd.isna(r) or math.isinf(r): 
            r = 0.0

    # NaN / inf entfernen
    if pd.isna(d) or pd.isna(t) or math.isinf(d) or math.isinf(t):
        continue

    dist[(i, j)] = float(d)
    risk[(i, j)] = float(r)
    time[(i, j)] = float(t)
    cost[(i, j)] = float(c) 

    

    nodes.add(i)
    nodes.add(j)

nodes = list(nodes)

print(f"Nodes: {len(nodes)} | Arcs: {len(dist)}")


# ============================================================
# 3. SETS
# ============================================================

df_vehicles = pd.read_csv(
    r"C:\Users\j.beckmann\OneDrive - Reply\Uni Jonas Beckmann\4.Semester\Operations Research\Projekt\Umwelftfreundliche_Routenplanung\data\processed\vehicles.csv"
)

DEPOT = "DEPOT"

if DEPOT not in nodes:
    nodes.append(DEPOT)

# Ladesäulen 
stations = [n for n in nodes if str(n).startswith("L")]

# Kunden
customers = [n for n in nodes if n != DEPOT and n not in stations]


all_nodes = customers + stations + [DEPOT]

vehicles = df_vehicles["type"].tolist()

# ============================================================
# FEHLENDE KANTEN (Penalty)
# ============================================================


max_dist = max(dist.values())
max_risk = max(risk.values()) if len(risk) > 0 else 1
max_time = max(time.values())
max_cost = max(cost.values()) 

BIG_PENALTY = 2

for i in nodes:
    for j in nodes:
        if i != j and (i, j) not in dist:

            dist[(i, j)] = max_dist * BIG_PENALTY
            risk[(i, j)] = max_risk * BIG_PENALTY
            time[(i, j)] = max_time * BIG_PENALTY
            cost[(i, j)] = max_cost * BIG_PENALTY 
# safety check
print("NaN Check Dist:", any(pd.isna(v) for v in dist.values()))
print("NaN Check Time:", any(pd.isna(v) for v in time.values()))
print("NaN Check Risk:", any(pd.isna(v) for v in risk.values()))

# ============================================================
# 4. PARAMETER
# ============================================================
df_kunden = pd.read_csv(r"C:\Users\j.beckmann\OneDrive - Reply\Uni Jonas Beckmann\4.Semester\Operations Research\Projekt\Umwelftfreundliche_Routenplanung\models\Data\OperationsResearch\kleine_instanz.csv")
df_kunden = df_kunden[df_kunden["id"] != "DEPOT"].copy()
df_kunden["quantity"] = pd.to_numeric(df_kunden["quantity"])

Demand = dict(zip(df_kunden["id"], df_kunden["quantity"]))

# Max Reichweite in km pro Fahrzeug 
MaxRange = dict(zip(df_vehicles["type"], df_vehicles["range_km"]))

# Ladezeit in Minuten 
RechargeTime = {s: 45 for s in stations}

Cap = dict(zip(df_vehicles["type"], df_vehicles["fuel_capacity_l"]))

FixCost = dict(zip(df_vehicles["type"], df_vehicles["fixcost"]))

VarCost = dict(zip(df_vehicles["type"], df_vehicles["variable_cost_per_km"]))

ServiceTime = {c: 30 for c in customers}

MAX_TIME = 480

ChargeCost = 50

w_cost = 0.3
w_risk = 0.7

# ============================================================
# 5. MODELL
# ============================================================

model = pl.LpProblem("Hazmat_VRP_Robust", pl.LpMinimize)


x = pl.LpVariable.dicts(
    "x",
    [(i, j, v) for i in all_nodes for j in all_nodes if i != j for v in vehicles],
    cat="Binary"
)

# Ankunftszeiten
T = pl.LpVariable.dicts(
    "T",
    [(i, v) for i in all_nodes for v in vehicles],
    lowBound=0
)

# Besuchte Kunden
U = pl.LpVariable.dicts(
    "U",
    customers,
    lowBound=0,
    upBound=len(customers)
)


# Batteriestand des Fahrzeugs v bei Ankunft an Knoten i
B = pl.LpVariable.dicts(
    "B",
    [(i, v) for i in all_nodes for v in vehicles],
    lowBound=0
)

# ============================================================
# 6. ZIELFUNKTION
# ============================================================

# Wir minimieren die gewichtete Summe aus Distanz, Risiko und variablen Kosten.
travel_cost = pl.lpSum(
    (w_cost * dist[(i, j)] + 
     w_risk * risk[(i, j)] +
     VarCost[v] * dist[(i, j)]
    ) * x[(i, j, v)]
    for i in nodes for j in nodes if i != j for v in vehicles
)

# Fixkosten: Wenn ein Fahrzeug v eingesetzt wird, fallen Fixkosten an.
fixed_cost = pl.lpSum(
    FixCost[v] * pl.lpSum(x[(DEPOT, j, v)] for j in customers + stations) # GEÄNDERT: stations ergänzt!
    for v in vehicles
)

# Ladezeit-Penalty (Wenn Station s angefahren wird, kostet das w_time * RechargeTime)
charge_penalty = pl.lpSum(
    (ChargeCost + w_time * RechargeTime[s]) * x[(i, s, v)]
    for i in all_nodes for s in stations for v in vehicles if i != s
)

# Zielfunktion um Ladezeit ergänzen
model += travel_cost + fixed_cost + charge_penalty

# ============================================================
# 7. CONSTRAINTS
# ============================================================

# Jeder Kunde genau einmal
for j in customers:
    model += pl.lpSum(
        x[(i, j, v)]
        for i in nodes if i != j for v in vehicles
    ) == 1


# Depot
for v in vehicles:
    model += (
        pl.lpSum(x[(DEPOT, j, v)] for j in customers + stations) ==
        pl.lpSum(x[(i, DEPOT, v)] for i in customers + stations)
    )

    model += pl.lpSum(x[(DEPOT, j, v)] for j in customers + stations) <= 1

# Ladestationen sind optional (können 0 oder 1 mal besucht werden)
for s in stations:
    model += pl.lpSum(
        x[(i, s, v)]
        for i in all_nodes if i != s for v in vehicles
    ) <= 1  

# Fluss 
for v in vehicles:
    for h in customers + stations:
        model += (
            pl.lpSum(x[(i, h, v)] for i in nodes if i != h) ==
            pl.lpSum(x[(h, j, v)] for j in nodes if j != h)
        )

# ============================================================
# 8. ZEIT
# ============================================================

BIG_M = 100000

# Start (Fahrt vom Depot zu Kunde oder Station)
for v in vehicles:
    for j in customers + stations:
        model += (
            T[(j, v)] >= time[(DEPOT, j)]
            - BIG_M * (1 - x[(DEPOT, j, v)])
        )

# Fahrt zwischen Knoten (Kunde/Station → Kunde/Station/Depot)
for v in vehicles:
    for i in customers + stations:
        for j in all_nodes:
            if i != j:
                # Dauer des Aufenthalts am Knoten i bestimmen
                stay_time = ServiceTime[i] if i in customers else RechargeTime[i]
                
                model += (
                    T[(j, v)] >=
                    T[(i, v)] + stay_time + time[(i, j)]
                    - BIG_M * (1 - x[(i, j, v)])
                )

# Max Tourzeit
for v in vehicles:
    model += pl.lpSum(
        time[(i, j)] * x[(i, j, v)]
        for i in nodes for j in nodes if i != j
    ) <= MAX_TIME


# ===========================================================
# Ladung
# ===========================================================

Load = pl.LpVariable.dicts("Load", [(i, v) for i in nodes for v in vehicles], lowBound=0)

for v in vehicles:
    model += Load[(DEPOT, v)] == Cap[v]

for v in vehicles:
    for i in nodes:
        for j in customers + stations:  # GEÄNDERT: stations integriert
            if i != j:
                # An Ladestationen wird KEIN Gefahrgut entladen
                dem_j = Demand[j] if j in customers else 0 
                model += (
                    Load[(j, v)] <=
                    Load[(i, v)] - dem_j + BIG_M * (1 - x[(i, j, v)])
                )

for v in vehicles:
    for j in customers + stations:
        model += Load[(j, v)] >= 0

# ============================================================
# 8.5 BATTERIE & REICHWEITE (E-VRP)
# ============================================================


for v in vehicles:
    for i in all_nodes:
        model += B[(i, v)] <= MaxRange[v]

for v in vehicles:
    model += B[(DEPOT, v)] == MaxRange[v]

for v in vehicles:
    for i in all_nodes:
        for j in all_nodes:
            if i != j:
                # Sicherstellen, dass die Kante existiert (sonst 9999)
                d_ij = dist.get((i, j), 9999)

                # Bedingung 1: LKW muss den Knoten j erreichen können (ohne Liegenbleiben)
                model += (
                    B[(i, v)] - d_ij >= -BIG_M * (1 - x[(i, j, v)])
                )
                
                # Bedingung 2: Batteriestand nach Ankunft
                if j in customers:
                    # An Kunden nimmt die Batterie um die Distanz ab
                    model += (
                        B[(j, v)] <= B[(i, v)] - d_ij + BIG_M * (1 - x[(i, j, v)])
                    )
                elif j in stations:
                    # An Ladesäulen wird die Batterie auf MaxRange aufgeladen.
                    # Das heißt, bei Abfahrt von j ist die Batterie voll.
                    model += (
                        B[(j, v)] == MaxRange[v] + BIG_M * (1 - x[(i, j, v)])
                    )

# ============================================================
# 9 WARM START (HEURISTIK STARTLÖSUNG)
# ============================================================

# PLATZHALTER: Später für den WarmStart des Routers 
# Format: Ein Dictionary mit der Fahrzeug-ID als Key und der Route als Liste.
heuristic_routes = {
    # "Truck1": ["DEPOT", "C8", "C7", "DEPOT"],
    # "Truck2": ["DEPOT", "C6", "C5", "C3", "C4", "C1", "C10", "DEPOT"],
    # "Truck3": ["DEPOT", "C2", "C9", "DEPOT"]
}

if heuristic_routes:
    print("Heuristik Lösung wird verwendet für Warm Start") 
    
    # 1. Alle Kanten initial auf 0 setzen (optional, aber sauberer)
    for i in nodes:
        for j in nodes:
            if i != j:
                for v in vehicles:
                    x[(i, j, v)].setInitialValue(0)
                    
    # 2. Die Heuristik-Kanten auf 1 setzen
    for v, route in heuristic_routes.items():
        for k in range(len(route) - 1):
            i = route[k]
            j = route[k+1]
            if (i, j, v) in x:
                x[(i, j, v)].setInitialValue(1)

# ============================================================
# 10. SOLVER (MIT WARM START)
# ============================================================

print("Starte Optimierung...")

# NEU: warmStart=True hinzugefügt, timeLimit auf 5 Stunden (18000 Sekunden)
solver = pl.PULP_CBC_CMD(
    msg=1, 
    timeLimit=300, 
    gapRel=0.02, 
    warmStart=True 
)

model.solve(solver)

print("\nStatus:", pl.LpStatus[model.status])
print("Objective:", pl.value(model.objective))

# ============================================================
# 11. ROUTEN
# ============================================================

def build_route(v):
    route = [DEPOT]
    current = DEPOT
    visited = set()

    while True:
        next_nodes = [
            j for j in all_nodes
            if j != current and pl.value(x[(current, j, v)]) is not None and pl.value(x[(current, j, v)]) > 0.5
        ]

        if not next_nodes:
            break

        nxt = next_nodes[0]
        route.append(nxt)

        if nxt == DEPOT:
            break

        if nxt in visited:
            route.append("SUBTOUR")
            break

        visited.add(nxt)
        current = nxt

    return route

print("\nROUTEN:")

for v in vehicles:
    r = build_route(v)

    if len(r) > 1:
        print(f"{v}: {' -> '.join(r)}")
    else:
        print(f"{v}: nicht genutzt")

#Kosten ausgeben 
print("\nKostenübersicht:")
total_travel_cost = pl.value(travel_cost)
total_fixed_cost = pl.value(fixed_cost)
total_charge_time_penalty = pl.value(charge_penalty)

Anzahl finaler Kanten nach Filter: 170
Nodes: 32 | Arcs: 60
NaN Check Dist: False
NaN Check Time: False
NaN Check Risk: False
Starte Optimierung...


c:\Users\j.beckmann\AppData\Local\anaconda3\Lib\site-packages\pulp\apis\coin_api.py:112: UserWarning: When using CBC on Windows, warmStart requires keepFiles=True.
  warnings.warn(



Status: Optimal
Objective: 5455.885174707752

ROUTEN:
MAN_eTGX: DEPOT -> L562 -> L41 -> C1 -> L162 -> C5 -> L147 -> DEPOT
Volvo_FH_Electric: DEPOT -> C6 -> L5 -> C7 -> C10 -> L326 -> DEPOT
Mercedes_eActros_600: DEPOT -> C4 -> C8 -> L13 -> C3 -> DEPOT

Kostenübersicht:


In [ ]:
# ============================================================
# PLAUSIBILITÄTSCHECKS
# ============================================================

print("\n--- PLAUSIBILITÄTSCHECKS ---")

# 1. Distanz > 0
dist_values = list(dist.values())
print("Min Distanz:", min(dist_values))
print("Max Distanz:", max(dist_values))

if any(d <= 0 for d in dist_values):
    print("Ungültige Distanzen (<=0 gefunden)")
else:
    print("Alle Distanzen positiv")

# ------------------------------------------------------------

# 2. Zeit konsistent mit Distanz (Geschwindigkeit)
speeds = []

for (i, j) in dist:
    d = dist[(i, j)]       # km
    t = time[(i, j)]       # Minuten
    
    if t > 0:
        speed = d / (t / 60)  # km/h
        speeds.append(speed)

print(f"Geschwindigkeit: min={min(speeds):.1f}, max={max(speeds):.1f} km/h")

if min(speeds) < 5 or max(speeds) > 130:
    print("Unplausible Geschwindigkeit")
else:
    print("Geschwindigkeit plausibel")

# ------------------------------------------------------------

# 3. Risiko-Werte prüfen
risk_values = list(risk.values())

print("Min Risiko:", min(risk_values))
print("Max Risiko:", max(risk_values))

if any(r < 0 for r in risk_values):
    print("Negatives Risiko gefunden")
else:
    print("Risiko plausibel")

# ------------------------------------------------------------

# 4. Nachfrage vs Kapazität
total_demand = sum(Demand.values())
total_capacity = sum(Cap[v] for v in vehicles)

print("Gesamtnachfrage:", total_demand)
print("Gesamtkapazität:", total_capacity)

if total_demand > total_capacity:
    print("Nachfrage > Kapazität → Problem infeasible")
else:
    print("Kapazität ausreichend")

# ------------------------------------------------------------

# 5. Zeitfenster grob machbar?
max_trip_time = max(time.values())

print("Max Einzelkante Zeit:", max_trip_time)
print("MAX_TIME:", MAX_TIME)

if max_trip_time > MAX_TIME:
    print("Einzelverbindung länger als MAX_TIME")
else:
    print("Zeitstruktur plausibel")

# ------------------------------------------------------------

# 6. Erreichbarkeit prüfen (wichtiger Check!)
unreachable = od_merged[od_merged["reachable"] == False]

if len(unreachable) > 0:
    print(f"{len(unreachable)} nicht erreichbare Relationen")
else:
    print("Alle Relationen erreichbar")

# ------------------------------------------------------------

# 7. Depot erreichbar?
for c in customers:
    if (DEPOT, c) not in dist or (c, DEPOT) not in dist:
        print(f"Depot nicht verbunden mit {c}")

print("Depot-Verbindungen geprüft")


--- PLAUSIBILITÄTSCHECKS ---
Min Distanz: 27.056752126615446
Max Distanz: 542.4528392498312
Alle Distanzen positiv
Geschwindigkeit: min=44.3, max=78.7 km/h
Geschwindigkeit plausibel
Min Risiko: 0.0
Max Risiko: 0.261319641015014
Risiko plausibel
Gesamtnachfrage: 57000
Gesamtkapazität: 105000
Kapazität ausreichend
Max Einzelkante Zeit: 513.5335312539524
MAX_TIME: 2000
Zeitstruktur plausibel
19 nicht erreichbare Relationen
Depot-Verbindungen geprüft


Hazmat VRP mit Reload, Risiko und Zeitrestriktionen

In [ ]:
# import pandas as pd
# import pulp as pl
# import math

# # ============================================================
# # 1. OD MATRIX LADEN
# # ============================================================

# od_df = pd.read_csv(
#     r"C:\Users\j.beckmann\OneDrive - Reply\Uni Jonas Beckmann\4.Semester\Operations Research\Projekt\Umwelftfreundliche_Routenplanung\models\Data\OperationsResearch\od_matrix_hazmat_small.csv"
# )

# od_df = od_df[od_df["profile"] == "safest"]

# # ============================================================
# # 2. DATENSTRUKTUREN
# # ============================================================

# dist, risk, time = {}, {}, {}
# nodes = set()

# for _, row in od_df.iterrows():

#     i, j = row["from"], row["to"]

#     d = row.get("dist_km", None)
#     r = row.get("risk_cost", 0)
#     t = row.get("time_min", None)

#     if pd.isna(d) or pd.isna(t) or math.isinf(d) or math.isinf(t):
#         continue

#     if pd.isna(r) or math.isinf(r):
#         r = 0

#     dist[(i, j)] = float(d)
#     risk[(i, j)] = float(r)
#     time[(i, j)] = float(t)

#     nodes.add(i)
#     nodes.add(j)

# nodes = list(nodes)

# # ============================================================
# # FEHLENDE KANTEN
# # ============================================================

# max_dist = max(dist.values())
# max_risk = max(risk.values()) if len(risk) > 0 else 1
# max_time = max(time.values())

# BIG_PENALTY = 2

# for i in nodes:
#     for j in nodes:
#         if i != j and (i, j) not in dist:
#             dist[(i, j)] = max_dist * BIG_PENALTY
#             risk[(i, j)] = max_risk * BIG_PENALTY
#             time[(i, j)] = max_time * BIG_PENALTY

# # ============================================================
# # SETS
# # ============================================================

# DEPOT = "DEPOT"
# customers = [n for n in nodes if n != DEPOT]

# vehicles = ["Truck1", "Truck2", "Truck3"]

# # ============================================================
# # PARAMETER
# # ============================================================

# Demand = {c: 2000 for c in customers}

# Cap = {
#     "Truck1": 10000,
#     "Truck2": 12000,
#     "Truck3": 8000
# }

# FixCost = { 
#     "Truck1": 1000,
#     "Truck2": 1200,
#     "Truck3": 1500
#     }

# ServiceTime = {c: 5 for c in customers}

# MAX_TIME = 800

# w_cost = 0.3
# w_risk = 0.7

# # ============================================================
# # MODELL
# # ============================================================

# model = pl.LpProblem("Hazmat_VRP_Reload", pl.LpMinimize)

# # Routing
# x = pl.LpVariable.dicts(
#     "x",
#     [(i, j, v) for i in nodes for j in nodes if i != j for v in vehicles],
#     cat="Binary"
# )

# # Zeit
# T = pl.LpVariable.dicts(
#     "T",
#     [(i, v) for i in customers for v in vehicles],
#     lowBound=0
# )

# # Subtour
# U = pl.LpVariable.dicts(
#     "U",
#     customers,
#     lowBound=0,
#     upBound=len(customers)
# )

# # Load
# Load = pl.LpVariable.dicts(
#     "Load",
#     [(i, v) for i in nodes for v in vehicles],
#     lowBound=0
# )

# # ============================================================
# # ZIELFUNKTION
# # ============================================================

# travel_cost = pl.lpSum(
#     (w_cost * dist[(i, j)] + w_risk * risk[(i, j)]) * x[(i, j, v)]
#     for i in nodes for j in nodes if i != j for v in vehicles
# )

# fixed_cost = pl.lpSum(
#     FixCost[v] * pl.lpSum(x[(DEPOT, j, v)] for j in customers)
#     for v in vehicles
# )

# model += travel_cost + fixed_cost

# # ============================================================
# # CONSTRAINTS
# # ============================================================

# # Jeder Kunde genau einmal
# for j in customers:
#     model += pl.lpSum(
#         x[(i, j, v)]
#         for i in nodes if i != j for v in vehicles
#     ) == 1

# # Fluss
# for v in vehicles:
#     for h in customers:
#         model += (
#             pl.lpSum(x[(i, h, v)] for i in nodes if i != h) ==
#             pl.lpSum(x[(h, j, v)] for j in nodes if j != h)
#         )

# # Depot: mehrere Starts erlaubt ✅
# for v in vehicles:
#     model += (
#         pl.lpSum(x[(DEPOT, j, v)] for j in customers) ==
#         pl.lpSum(x[(i, DEPOT, v)] for i in customers)
#     )

# # ============================================================
# # LOAD LOGIK (WICHTIG)
# # ============================================================

# BIG_M = 100000

# for v in vehicles:
#     # Start im Depot: volle Ladung
#     model += Load[(DEPOT, v)] == Cap[v]

#     for i in nodes:
#         for j in customers:
#             if i != j:
#                 # Verbrauch bei Kunde
#                 model += (
#                     Load[(j, v)] <= Load[(i, v)] - Demand[j]
#                     + BIG_M * (1 - x[(i, j, v)])
#                 )

#                 # Nicht negativ
#                 model += Load[(j, v)] >= 0

#     # Reload im Depot
#     for i in customers:
#         model += (
#             Load[(DEPOT, v)] >= Load[(i, v)]
#             + BIG_M * (x[(i, DEPOT, v)] - 1)
#         )

# # ============================================================
# # ZEIT
# # ============================================================

# for v in vehicles:
#     for j in customers:
#         model += (
#             T[(j, v)] >= time[(DEPOT, j)]
#             - BIG_M * (1 - x[(DEPOT, j, v)])
#         )

# for v in vehicles:
#     for i in customers:
#         for j in customers:
#             if i != j:
#                 model += (
#                     T[(j, v)] >=
#                     T[(i, v)] + ServiceTime[i] + time[(i, j)]
#                     - BIG_M * (1 - x[(i, j, v)])
#                 )

# for v in vehicles:
#     for i in customers:
#         model += (
#             T[(i, v)] + ServiceTime[i] + time[(i, DEPOT)]
#             <= MAX_TIME + BIG_M * (1 - x[(i, DEPOT, v)])
#         )

# for v in vehicles:
#     model += pl.lpSum(
#         time[(i, j)] * x[(i, j, v)]
#         for i in nodes for j in nodes if i != j
#     ) <= MAX_TIME

# # ============================================================
# # MTZ
# # ============================================================

# for i in customers:
#     for j in customers:
#         if i != j:
#             model += (
#                 U[i] - U[j]
#                 + len(customers) * pl.lpSum(x[(i, j, v)] for v in vehicles)
#                 <= len(customers) - 1
#             )

# # ============================================================
# # SOLVER
# # ============================================================

# solver = pl.PULP_CBC_CMD(msg=1, timeLimit=300, gapRel=0.02)

# model.solve(solver)

# print("\nStatus:", pl.LpStatus[model.status])
# print("Objective:", pl.value(model.objective))

# # ============================================================
# # ROUTEN
# # ============================================================


# def extract_routes_for_vehicle(v):
#     routes = []

#     # aktive Kanten sammeln
#     edges = [
#         (i, j)
#         for i in nodes for j in nodes if i != j
#         if pl.value(x[(i, j, v)]) is not None and pl.value(x[(i, j, v)]) > 0.5
#     ]

#     # Nachfolger-Mapping
#     successors = {}
#     for (i, j) in edges:
#         if i not in successors:
#             successors[i] = []
#         successors[i].append(j)

#     # alle Touren vom Depot starten
#     while True:

#         if DEPOT not in successors or len(successors[DEPOT]) == 0:
#             break

#         route = [DEPOT]
#         current = DEPOT

#         while True:
#             if current not in successors or len(successors[current]) == 0:
#                 break

#             nxt = successors[current].pop(0)
#             route.append(nxt)

#             if nxt == DEPOT:
#                 break

#             current = nxt

#         if len(route) > 2:
#             routes.append(route)

#     return routes


# print("\nROUTEN:")

# for v in vehicles:
#     routes = extract_routes_for_vehicle(v)

#     if len(routes) == 0:
#         print(f"{v}: nicht genutzt")
#     else:
#         for idx, r in enumerate(routes, 1):
#             print(f"{v} Tour {idx}: {' -> '.join(r)}")

c:\Users\j.beckmann\AppData\Local\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\j.beckmann\AppData\Local\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (



Status: Optimal
Objective: 4312.362609529095

ROUTEN:
Truck1 Tour 1: DEPOT -> C3 -> C4 -> C5 -> C6 -> C7 -> DEPOT
Truck2 Tour 1: DEPOT -> C2 -> C9 -> DEPOT
Truck3 Tour 1: DEPOT -> C8 -> C10 -> C1 -> DEPOT
